# KV Cache Lab：Radix 树完整性与 SGLang 的 LFU 淘汰

目标是先通过分支、节点拆分、锁定和淘汰理解 mini 的 radix 树，亲手补一个很小的树完整性功能，再研究资源账本并实现主仓库的 LFU 淘汰。实现由你完成；这里给可观察要求、负载、问题和阅读入口，不给补丁。


## 为什么选这个范围

mini 已经有前缀匹配、节点拆分、锁定、page 分配和 LRU 淘汰，但 radix 树的 `check_integrity` 尚无行为。主仓库有更完整的缓存生命周期与可配置淘汰策略。两个实现题都落在现有缓存对象上，避免引入多级 KV、host cache 或新 allocator。

| 阶段 | 你要做的事 | 验收 |
|---|---|---|
| 0. 观察树 | 预测共享前缀、分支、锁定、页对齐和淘汰的状态 | 现有代码应通过 |
| 1. 树内小改动 | 让 `check_integrity` 识别正常树和两类损坏 | 健康树通过，计数与拓扑损坏被拒绝 |
| 2. 对照魔改 | 暂时让默认淘汰偏向最近访问的分支，再恢复 | 同一负载选出的分支改变 |
| 3. 资源账本 | 追踪 `CacheManager` 的 page 分配与回收 | 现有行为测试通过 |
| 4. 主仓库功能 | 默认仍为 LRU，允许显式选择 LFU | 两种策略选择不同分支；锁定始终有效 |

请保留现有缓存测试。Phase 2 的临时改动在进入 Phase 4 前恢复；Phase 4 的实现应可被离线 `LLM` 配置使用。


## 先分清三层

| mini 文件 | 读它要回答的问题 |
|---|---|
| `python/minisgl/kvcache/mha_pool.py` | 物理 K/V 张量在哪里？ |
| `python/minisgl/scheduler/cache.py`、`table.py` | 空闲 page、请求的 page table 和前缀树怎样交接？ |
| `python/minisgl/kvcache/radix_cache.py` | token 前缀与可复用 page index 怎样组织、保护和淘汰？ |
| `python/minisgl/scheduler/prefill.py`、`scheduler.py` | 请求何时查询/插入缓存、何时释放资源？ |

建议先读 `radix_cache.py` 的公开方法，再顺着 `CacheManager` 的调用链回到 scheduler。`kernel/radix.py` 的比较实现、attention backend、CUDA graph、模型代码和主仓库 HiCache 暂时不用读。


## 环境

前面的缓存结构与策略负载只需 CPU，但要在**已安装 mini-sglang 及其依赖**的环境、从仓库根目录运行；首次运行可能编译项目的 CPU 前缀比较扩展。完整 `LLM` 接入检查需 Linux + NVIDIA CUDA 服务器。`observe.py` 只驱动真实缓存对象和断言，不替你改核心代码。


## Phase 0：先画出真正的 radix 树

画出 `insert_prefix` → `match_prefix` → `lock_handle` → `evict` → `unlock`。分别标出 token key、page indices、压缩边、树节点、handle 与 `evictable_size/protected_size` 由谁持有。

运行前预测：

1. 插入 `[11,12,13,14]` 与 `[11,12,15,16]`，两条分支的前两个 page indices 相同。树保存几个 token/page index？为什么不是八个？
2. 查询 `[11,12,99]` 后树是否可能改变？锁住返回的 handle，再淘汰四个 token，哪些 page indices 可以回收？解锁后又能回收什么？
3. `page_size=4` 时插入六个 token，能缓存的长度是多少？两个只在第四个 token 不同的输入能共享多少页？

请把每次操作后的树画成“边上的 token 片段 → page index 片段”，并记录每个节点的锁定状态；仅画命中长度不算完成。


In [ ]:
%%bash
python lab/kv_cache/observe.py --case radix-tree
python lab/kv_cache/observe.py --case split-lock
python lab/kv_cache/observe.py --case page


**记录**：把三次运行的命中长度、size 计数与回收的 page indices 记下来。解释分裂后共享前缀为何只计一次，以及锁住共享节点时两个尾部分支为何仍可淘汰。再追问：`match_prefix` 的接口文字与实际树结构变化完全一致吗？使用返回的 page indices 前为什么需要锁定 handle？


## Phase 1：给 radix 树补一个结构完整性实验

`RadixPrefixCache.check_integrity()` 是 mini 已有的接口。请让它在正常的插入、节点拆分、锁定、解锁与淘汰后通过；若树的总量账本被人为改坏，或父子拓扑被人为断开，必须抛出错误。只检查 radix 树自身可以判断的事实；不要在这里重写 `CacheManager` 的空闲 page 检查，也不要改变插入/匹配/淘汰语义。

先运行下面的负载：它先构造共享前缀的分支，再锁住共享前缀，分别注入两种损坏，最后恢复并继续淘汰。未实现时，它应在第一种损坏上失败。你的实现让整个负载通过后，再重跑 Phase 0。

**先答后改**：压缩边的 key/value 长度有何关系？根节点和普通节点的计数含义是否相同？如何让一个分支的锁定不被误算为整个子树的保护？如果只比较 `size_info.total_size`，能否发现父子关系断裂？

阅读入口：mini 的 `kvcache/base.py` 接口契约、`kvcache/radix_cache.py` 的节点生命周期与公开方法、`scheduler/cache.py` 对完整性检查的调用。主仓库只对照 [RadixCache](https://github.com/sgl-project/sglang/blob/main/python/sglang/srt/mem_cache/radix_cache.py) 的职责边界。暂时不用读 attention kernel、HiCache 和分布式代码。


In [ ]:
%%bash
python lab/kv_cache/observe.py --case integrity
python lab/kv_cache/observe.py --case radix-tree


## Phase 2：一次可恢复的极小魔改

下面的负载依次插入 A、B 两条独立分支，随后再访问 A，最后只淘汰一条分支。先运行默认行为，再暂时把默认淘汰规则改成**优先淘汰最近访问的可淘汰分支**；其余缓存语义、锁与 page 回收行为保持不变。让第二个命令通过后，恢复默认行为，并重新运行第一个命令。

动手前回答：插入顺序与最后访问顺序在这个负载中是否相同？如果相同，负载能否区分 LRU、MRU 与按插入顺序淘汰？淘汰内部节点与叶子节点的条件相同吗？

阅读入口：mini 的 radix 树节点与淘汰流程。不要为这个临时改动新增通用策略框架。


In [ ]:
%%bash
# 改动前以及恢复后
python lab/kv_cache/observe.py --case eviction --expect lru


In [ ]:
%%bash
# 临时魔改期间
python lab/kv_cache/observe.py --case eviction --expect mru


## Phase 3：理解 CacheManager 的资源账本

在 mini 中，前缀树决定哪些缓存分支可回收；`CacheManager` 持有空闲 page 并按需请求淘汰。请从 `CacheManager._allocate` 追到 radix `evict`，再从请求完成路径追到 `cache_req` 和 `lazy_free_region`。

问题：树里的 `evictable_size` 与 `CacheManager.available_size` 为什么不是同一个数字？一次淘汰若回收了多于请求数量的 token，空闲 page 账本该怎样变化？`page_size > 1` 时哪些值必须保持对齐？先运行现有行为测试，再解释其中一次 allocate→evict→allocate 的账本。


In [ ]:
%%bash
python -m pytest -q -o addopts= tests/core/test_cache_allocate.py


## Phase 4：新增主仓库的 LFU 淘汰能力

**外部行为要求**：默认策略仍按原有 LRU 行为工作；显式选择 `lfu` 时，优先淘汰复用次数较少的可淘汰分支，次数相同时保持稳定的先后规则。本实验把对已缓存前缀的再次 `insert_prefix` 视为一次复用。验收负载先插入 A、B，然后多次复用 A，最后一次复用 B：B 最近被访问，但 A 复用更多。默认策略淘汰 A，LFU 淘汰 B。已经锁住的分支在两种策略下都不能被淘汰；淘汰后剩余分支仍可匹配，缓存大小与 page 账本保持正确。

实验至少要支持 `RadixPrefixCache(..., eviction_policy="lfu")`，并能从离线 `LLM(..., radix_eviction_policy="lfu")` 选用它。只要求 `lru` 和 `lfu`；不新增完整的策略层次结构。

先答再改：哪些操作应被算作一次前缀复用？同一请求的 chunked prefill 会不会重复计数？节点拆分后，新旧分支的复用历史应如何解释？配置从 `LLM` 到缓存实例经过哪些已有层？主仓库如何将策略选择与缓存类型区分？


### Phase 4 的行为验收

前三个命令分别检查默认 LRU、LFU 和“锁定优先于淘汰策略”。第四个命令在 `CacheManager` 的 page 不足时触发淘汰，并核对回收 page 与账本；第五个命令检查此前的节点拆分与锁定行为没有回归。新增功能未实现时，带 `lfu` 的命令会失败；完成后应通过。不要只检查配置字段是否存在。


In [ ]:
%%bash
python lab/kv_cache/observe.py --case frequency
python lab/kv_cache/observe.py --case frequency --policy lfu
python lab/kv_cache/observe.py --case locked-frequency --policy lfu
python lab/kv_cache/observe.py --case manager
python lab/kv_cache/observe.py --case split-lock


### 配置路径检查（Linux CUDA 服务器）

完成 CPU 行为验收后，用已缓存的模型 config/tokenizer 跑一个离线请求，确认新选项可以从 `LLM` 进入实际缓存构造路径。这个 smoke test 只验证配置接入；策略是否真的改变淘汰对象，以上面的 CPU 行为验收为准。


In [ ]:
import os
import platform
import torch
from minisgl.core import SamplingParams
from minisgl.llm import LLM

assert platform.system() == "Linux" and torch.cuda.is_available()
model = os.environ.get("MODEL_PATH", "Qwen/Qwen3-0.6B")
llm = LLM(model, use_dummy_weight=True, max_seq_len_override=512, radix_eviction_policy="lfu")
result = llm.generate([[100, 101, 102, 103]], SamplingParams(ignore_eos=True, max_tokens=2))
assert len(result) == 1 and len(result[0]["token_ids"]) == 2


## 对照 SGLang 主仓库

按这个顺序看：[公开选项](https://github.com/sgl-project/sglang/blob/main/python/sglang/srt/arg_groups/choices.py) → [淘汰策略](https://github.com/sgl-project/sglang/blob/main/python/sglang/srt/mem_cache/evict_policy.py) → [RadixCache](https://github.com/sgl-project/sglang/blob/main/python/sglang/srt/mem_cache/radix_cache.py) → [缓存初始化参数](https://github.com/sgl-project/sglang/blob/main/python/sglang/srt/mem_cache/cache_init_params.py)。只追 LRU/LFU、前缀复用、节点生命周期和策略配置。主仓库额外支持的策略、host cache、KV 事件、分布式通信先跳过。

复盘：mini 只支持两种策略时，哪些主仓库结构没有必要照搬？若以后要加更复杂的策略，你当前设计在哪些地方会开始不够用？


## 留作问题，不扩大本次实现

mini 的 `RadixPrefixCache.reset` 仍值得研究：

- 如果只清空 radix 树，已占用的 KV page 会由谁归还？在有活动请求持有 handle 时，清空树会发生什么？
- 主仓库的 HiCache、cache salt、TTL 或 session protection 会改变哪一层的契约？为什么这次不把它们塞进 LFU 实验？

写出问题和最小反例即可，本次不实现这些能力。


## 交付自查

保留 Phase 0 的树形图、Phase 1 完整性检查失败与通过轨迹、Phase 2 改前/改中/恢复后的三份淘汰轨迹，以及 LFU 的通过轨迹。运行本 lab 的全部 CPU 命令和现有 `tests/core/test_cache_allocate.py`；有 Linux CUDA 环境时再跑配置 smoke test。记录一次缓存命中、一次节点拆分、一次锁保护、一次淘汰及其 page 账本变化。
